# 토큰 임베딩: 구조와 분석 - 임베딩 공간의 기하학

- Tutorial ID: `tut-10b-1`
- Tutorial: 토큰 임베딩: 구조와 분석
- Section ID: `s10b-1-1`
- Section: 임베딩 공간의 기하학


In [ ]:
# ============================================================
# 코드 읽는 법 — 임베딩 공간의 기하학
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 55)
print("토큰 임베딩 공간 분석")
print("=" * 55)

np.random.seed(42)

vocab = ['<BOS>', 'cat', 'cats', 'dog', 'dogs', 
         'king', 'queen', 'man', 'woman', 'the', 'a', '<EOS>']
V = len(vocab)
d_model = 8

# 임베딩 행렬 (실제처럼 시뮬레이션)
W_E = np.random.randn(V, d_model) * 0.5

# 의미적 유사성을 시뮬레이션하기 위해 유사 토큰끼리 유사한 벡터 부여
# cat와 cats는 유사
W_E[vocab.index('cats')] = W_E[vocab.index('cat')] + np.random.randn(d_model) * 0.1
# dog와 dogs는 유사  
W_E[vocab.index('dogs')] = W_E[vocab.index('dog')] + np.random.randn(d_model) * 0.1
# king과 queen은 비교적 유사
W_E[vocab.index('queen')] = W_E[vocab.index('king')] + np.random.randn(d_model) * 0.3

In [ ]:
# 1. 코사인 유사도 분석

In [ ]:
print("\n1. 코사인 유사도 행렬")
print("-" * 40)

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

# 상위 쌍들의 유사도
similar_pairs = [
    ('cat', 'cats'), ('dog', 'dogs'), ('king', 'queen'),
    ('man', 'woman'), ('cat', 'dog'), ('the', 'a')
]

print("  토큰 쌍           코사인 유사도")
print("  " + "-" * 35)
for w1, w2 in similar_pairs:
    i1, i2 = vocab.index(w1), vocab.index(w2)
    sim = cosine_similarity(W_E[i1], W_E[i2])
    bar = '█' * int((sim + 1) * 10)
    print(f"  ({w1:8s}, {w2:8s}): {bar} {sim:+.3f}")

In [ ]:
# 2. PCA로 임베딩 구조 분석

In [ ]:
print("\n2. PCA 분석 (임베딩 차원 활용)")
print("-" * 40)

# SVD = PCA (중심화 후)
W_E_centered = W_E - W_E.mean(axis=0)
U, S, Vt = np.linalg.svd(W_E_centered)

print(f"  특이값 스펙트럼:")
total_variance = np.sum(S**2)
cumulative = 0
for i, s in enumerate(S):
    cumulative += s**2
    pct = cumulative / total_variance * 100
    bar = '█' * int(s / S[0] * 20)
    print(f"  PC{i+1}: {bar} {s:.3f} (누적 분산: {pct:.1f}%)")
    if pct > 95:
        print(f"  → PC{i+1}까지 95% 이상 분산 설명!")
        break

In [ ]:
# 3. 가중치 묶기 (Weight Tying)

In [ ]:
print("\n3. 가중치 묶기 (Weight Tying)")
print("-" * 40)

W_U = np.random.randn(d_model, V) * 0.3

# 가중치 묶기: W_U = W_E^T
W_U_tied = W_E.T  # 완전 묶기

print("  가중치 묶기를 사용하는 이유:")
print(f"  - W_E 파라미터: {W_E.size:,}")
print(f"  - W_U 파라미터: {W_U.size:,}")
print(f"  - 별도 사용 시 총: {W_E.size + W_U.size:,}")
print(f"  - 묶기 사용 시 총: {W_E.size:,} (절반!)")
print()
print("  묶기의 의미:")
print("  - '입력 임베딩'과 '출력 임베딩'이 동일한 공간 사용")
print("  - 어떤 토큰을 '인식'하는 것과 '예측'하는 것이 같은 표현")
print("  - 많은 언어 모델(GPT-2 등)에서 사용")

# 묶기 시의 바이그램 테이블
bigram_table_tied = W_E @ W_U_tied  # = W_E @ W_E^T
print(f"\n  묶기 바이그램 테이블 = W_E @ W_E^T (그람 행렬)")
print(f"  이 행렬의 대각 원소 = ||W_E[i]||^2 (각 토큰의 자기 점수)")

self_scores = np.diag(bigram_table_tied)
print(f"  상위 자기 점수 토큰:")
top_self = np.argsort(self_scores)[-5:][::-1]
for i in top_self:
    print(f"    '{vocab[i]}': {self_scores[i]:.3f}")

In [ ]:
# 4. Word2Vec 유사 대수 관계

In [ ]:
print("\n4. 임베딩 대수 관계 (Word2Vec 스타일)")
print("-" * 40)
print("  king - man + woman ≈ queen ?")

king = W_E[vocab.index('king')]
man = W_E[vocab.index('man')]
woman = W_E[vocab.index('woman')]
queen = W_E[vocab.index('queen')]

analogy_vec = king - man + woman

# 가장 가까운 토큰 찾기
similarities = [(v, cosine_similarity(analogy_vec, W_E[i])) for i, v in enumerate(vocab)]
similarities.sort(key=lambda x: x[1], reverse=True)

print("  'king - man + woman'에 가장 가까운 토큰:")
for word, sim in similarities[:5]:
    print(f"    {word:10s}: {sim:+.3f}")

print("\n  (주의: 이 예시는 소규모 랜덤 임베딩으로 실제 언어 패턴이 없습니다)")
print("  실제 대형 언어 모델에서는 이런 의미적 대수 관계가 실제로 작동합니다!")